# Creating Your First Agent Society

You can also check this cookbook in colab [here](https://colab.research.google.com/drive/1cmWPxXEsyMbmjPhD2bWfHuhd_Uz6FaJQ?usp=sharing)

## Philosophical Bits



> *What magical trick makes us intelligent? The trick is that there is no trick. The power of intelligence stems from our vast diversity, not from any single, perfect principle.*
>
> -- Marvin Minsky, The Society of Mind, p. 308

In this section, we will take a spite of the task-oriented `RolyPlaying()` class. We design this in an instruction-following manner. The essence is that to solve a complex task, you can enable two communicative agents collabratively working together step by step to reach solutions. The main concepts include:

- **Task**: a task can be as simple as an idea, initialized by an inception prompt.

- **AI User**: the agent who is expected to provide instructions.

- **AI Assistant**: the agent who is expected to respond with solutions that fulfills the instructions.

## Quick Start
Let's first play with a `ChatAgent` instance by simply initialize it with a system message and interact with user messages.

### 🕹 Step 0: Prepartions

In [ ]:
!pip install "camel-ai==0.2.9"

### Setting Up API Keys

You'll need to set up your API keys for OpenAI.

In [ ]:
import os
from getpass import getpass

# Prompt for the API key securely
openai_api_key = getpass('Enter your API key: ')
os.environ["OPENAI_API_KEY"] = openai_api_key

In [1]:
# Import necessary classes
from camel.societies import RolePlaying
from camel.types import TaskType, ModelType, ModelPlatformType
from camel.configs import ChatGPTConfig
from camel.models import ModelFactory
import os
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType

model = ModelFactory.create(
    model_platform=ModelPlatformType.OPENAI_COMPATIBLE_MODEL,
    model_type="deepseek-chat",
    api_key=os.environ.get("OPENAI_COMPATIBILIY_API_KEY"),
    url=os.environ.get("OPENAI_COMPATIBILIY_API_BASE_URL"),
    model_config_dict={"temperature": 0.4, "max_tokens": 4096},
)


### 🕹 Step 1: Configure the Role-Playing Session


#### Set the `Task` Arguments

In [2]:
uploaddata="我是上传的资料"
task_kwargs = {
    'task_prompt': '基于用户上传的资料{uploaddata}生成大量的高质量问答数据对',
    'with_task_specify': True,
    'task_specify_agent_kwargs': {'model': model}
}

#### Set the `User` Arguments
You may think the user as the `instruction sender`.

In [3]:
user_role_kwargs = {
    'user_role_name': '有很多不一样的提问题角度，可以基于用户上传的资料提出各种问题',
    'user_agent_kwargs': {'model': model}
}

#### Set the `Assistant` Arguments
Again, you may think the assistant as the `instruction executor`.

In [4]:
assistant_role_kwargs = {
    'assistant_role_name': '善于基于用户上传的资料对各种问题进行详细回答',
    'assistant_agent_kwargs': {'model': model}
}

### Step 2: Kickstart Your Society
Putting them altogether – your role-playing session is ready to go!

In [5]:
society = RolePlaying(
    **task_kwargs,             # The task arguments
    **user_role_kwargs,        # The instruction sender's arguments
    **assistant_role_kwargs,   # The instruction receiver's arguments
)

### Step 3: Solving Tasks with Your Society
Hold your bytes. Prior to our travel, let's define a small helper function.

In [6]:
def is_terminated(response):
    """
    Give alerts when the session shuold be terminated.
    """
    if response.terminated:
        role = response.msg.role_type.name
        reason = response.info['termination_reasons']
        print(f'AI {role} terminated due to {reason}')

    return response.terminated

Time to chart our course – writing a simple loop for our society to proceed:

In [7]:
def run(society, round_limit: int=10):

    # Get the initial message from the ai assistant to the ai user
    input_msg = society.init_chat()

    # Starting the interactive session
    for _ in range(round_limit):

        # Get the both responses for this round
        assistant_response, user_response = society.step(input_msg)

        # Check the termination condition
        if is_terminated(assistant_response) or is_terminated(user_response):
            break

        # Get the results
        print(f'[AI User] {user_response.msg.content}.\n')
        # Check if the task is end
        if 'CAMEL_TASK_DONE' in user_response.msg.content:
            break
        print(f'[AI Assistant] {assistant_response.msg.content}.\n')



        # Get the input message for the next round
        input_msg = assistant_response.msg

    return None

In [8]:
run(society)

[AI User] Instruction: 请基于用户上传的资料，生成第一个高质量问答对，涵盖历史领域。
Input: None.

[AI Assistant] Solution: 

**Question:** 在古埃及，法老的权力是如何体现的？

**Answer:** 在古埃及，法老的权力主要通过宗教、政治和经济三个方面体现。首先，法老是神的化身，被视为太阳神拉的儿子，因此拥有神圣的地位。这种宗教上的神化使得法老在政治上具有至高无上的权威，能够制定法律、指挥军队和进行外交活动。其次，法老在政治上通过中央集权的行政体系来管理国家，设立各级官员来执行他的命令，确保国家的统一和稳定。最后，法老在经济上控制着国家的资源，包括土地、劳动力和财富，通过税收和贡品来维持国家的运转和军事力量。这种权力的集中和神圣化使得法老成为古埃及社会中无可争议的统治者。

Next request..

[AI User] Instruction: 请基于用户上传的资料，生成第二个高质量问答对，涵盖科学领域。
Input: None.

[AI Assistant] Solution: 

**Question:** 量子力学中的“叠加态”是什么意思？

**Answer:** 在量子力学中，“叠加态”是指一个量子系统可以同时处于多个状态的线性组合。具体来说，如果一个量子系统可以处于状态A和状态B，那么它也可以处于状态A和状态B的叠加态，即状态C = αA + βB，其中α和β是复数系数，满足|α|^2 + |β|^2 = 1。这意味着在测量之前，系统既不完全处于状态A，也不完全处于状态B，而是处于一种“模糊”的状态。只有当进行测量时，系统才会“坍缩”到其中一个状态，并且测量结果的概率由α和β的模平方决定。这种叠加态的概念是量子力学与经典物理学的一个重要区别，它解释了量子系统中的一些奇特现象，如双缝实验中的干涉条纹。

Next request..

[AI User] Instruction: 请基于用户上传的资料，生成第三个高质量问答对，涵盖文化领域。
Input: None.

[AI Assistant] Solution: 

**Question:** 中国传统节日“春节”有哪些重要的文化习俗？

**Answer:** 中国传统节日“春节”有许多重要的文化习俗，这些习俗反

KeyboardInterrupt: 